<a href="https://colab.research.google.com/github/Viswesh934/Google-ADK-Sample/blob/main/GoogleADK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install google-adk -q
!pip install litellm -q

print("Installation complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.0 MB/s eta 0:00:00
Installation complete.


In [ ]:
import os
import asyncio
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm # For multi-model support
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types # For creating message Content/Parts

import warnings
# Ignore all warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(level=logging.ERROR)

print("Libraries imported.")

Libraries imported.


In [ ]:
#import secret

from google.colab import userdata
os.environ["GOOGLE_API_KEY"] =userdata.get("GOOGLE_API_KEY_1")


In [ ]:
MODEL_GEMMA = "gemini-2.0-flash"

In [ ]:
def generate_smart_hashtags(phrase: str) -> dict:
    """Generates hashtags based on phrase mood and intent (mocked).

    Args:
        phrase (str): Input phrase to analyze.

    Returns:
        dict: Contains 'status', 'mood', 'intent', and 'hashtags'.
    """
    print(f"--- Smart Hashtag Generator called for: {phrase} ---")

    if not phrase.strip():
        return {"status": "error", "error_message": "Phrase cannot be empty."}

    phrase_lower = phrase.lower()

    # Mock mood detection
    mood_map = {
        "happy": ["fun", "joy", "celebrate", "sunny", "smile"],
        "sad": ["loss", "alone", "rainy", "heartbreak"],
        "motivational": ["grind", "focus", "goal", "dream", "hustle"],
    }

    # Mock intent detection
    intent_map = {
        "marketing": ["buy", "sale", "deal", "brand", "discount"],
        "personal": ["my day", "i feel", "today was", "my journey"],
        "informative": ["how to", "tutorial", "guide", "learn", "did you know"],
    }

    mood = next((m for m, words in mood_map.items() if any(w in phrase_lower for w in words)), "neutral")
    intent = next((i for i, triggers in intent_map.items() if any(t in phrase_lower for t in triggers)), "general")

    # Mock tag database
    tags_by_mood_intent = {
        ("happy", "personal"): ["#happyvibes", "#blessed", "#goodday"],
        ("motivational", "marketing"): ["#goalgetter", "#growth", "#successmindset"],
        ("sad", "personal"): ["#selfcare", "#mentalhealth", "#healing"],
        ("neutral", "informative"): ["#didyouknow", "#learnsomethingnew", "#infopost"],
    }

    hashtags = tags_by_mood_intent.get((mood, intent), [f"#{word}" for word in phrase_lower.split()])

    return {
        "status": "success",
        "mood": mood,
        "intent": intent,
        "hashtags": hashtags
    }

# Example Usage
print(generate_smart_hashtags("My journey to success is filled with hustle and grind."))
print(generate_smart_hashtags("Rainy days make me feel a bit alone."))
print(generate_smart_hashtags("Learn how to grow your brand with these tips!"))


--- Smart Hashtag Generator called for: My journey to success is filled with hustle and grind. ---
{'status': 'success', 'mood': 'motivational', 'intent': 'personal', 'hashtags': ['#my', '#journey', '#to', '#success', '#is', '#filled', '#with', '#hustle', '#and', '#grind.']}
--- Smart Hashtag Generator called for: Rainy days make me feel a bit alone. ---
{'status': 'success', 'mood': 'sad', 'intent': 'general', 'hashtags': ['#rainy', '#days', '#make', '#me', '#feel', '#a', '#bit', '#alone.']}
--- Smart Hashtag Generator called for: Learn how to grow your brand with these tips! ---
{'status': 'success', 'mood': 'neutral', 'intent': 'marketing', 'hashtags': ['#learn', '#how', '#to', '#grow', '#your', '#brand', '#with', '#these', '#tips!']}


In [ ]:
AGENT_MODEL=MODEL_GEMMA
# Define the hashtag generation agent

hashtag_agent = Agent(
    name="hashtag_agent_v1",
    model=AGENT_MODEL,  # Can be a string for Gemini or a LiteLlm object
    description="Generates relevant hashtags based on a given phrase.",
    instruction=(
        "You are a helpful social media assistant. "
        "When the user provides a phrase or topic, use the 'generate_hashtags' tool "
        "to generate relevant and engaging hashtags. "
        "If the tool returns an error, notify the user politely. "
        "If the tool succeeds, display the hashtags clearly in a list or comma-separated format."
    ),
    tools=[generate_smart_hashtags],  # Pass the function directly
)

print(f"Agent '{hashtag_agent.name}' created using model '{AGENT_MODEL}'.")



Agent 'hashtag_agent_v1' created using model 'gemini-2.0-flash'.


In [ ]:
session_service = InMemorySessionService()

# Define constants for identifying the interaction context
APP_NAME = "SMART_HASHTAGS"
USER_ID = "user_1"
SESSION_ID = "session_001" # Using a fixed ID for simplicity

# Create the specific session where the conversation will happen
session = session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)
print(f"Session created: App='{APP_NAME}', User='{USER_ID}', Session='{SESSION_ID}'")

# --- Runner ---
# Key Concept: Runner orchestrates the agent execution loop.
runner = Runner(
    agent=hashtag_agent, # The agent we want to run
    app_name=APP_NAME,   # Associates runs with our app
    session_service=session_service # Uses our session manager
)
print(f"Runner created for agent '{runner.agent.name}'.")

Session created: App='SMART_HASHTAGS', User='user_1', Session='session_001'
Runner created for agent 'hashtag_agent_v1'.


In [ ]:
# @title Define Agent Interaction Function

from google.genai import types # For creating message Content/Parts

async def call_agent_async(query: str, runner, user_id, session_id):
  """Sends a query to the agent and prints the final response."""
  print(f"\n>>> User Query: {query}")

  # Prepare the user's message in ADK format
  content = types.Content(role='user', parts=[types.Part(text=query)])

  final_response_text = "Agent did not produce a final response." # Default

  # Key Concept: run_async executes the agent logic and yields Events.
  # We iterate through events to find the final answer.
  async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content):
      # You can uncomment the line below to see *all* events during execution
      # print(f"  [Event] Author: {event.author}, Type: {type(event).__name__}, Final: {event.is_final_response()}, Content: {event.content}")

      # Key Concept: is_final_response() marks the concluding message for the turn.
      if event.is_final_response():
          if event.content and event.content.parts:
             # Assuming text response in the first part
             final_response_text = event.content.parts[0].text
          elif event.actions and event.actions.escalate: # Handle potential errors/escalations
             final_response_text = f"Agent escalated: {event.error_message or 'No specific message.'}"
          # Add more checks here if needed (e.g., specific error codes)
          break # Stop processing events once the final response is found

  print(f"<<< Agent Response: {final_response_text}")

In [ ]:
async def run_conversation():
    await call_agent_async(
        "Generate hashtags for travel photography.",
        runner=runner,
        user_id=USER_ID,
        session_id=SESSION_ID
    )

    await call_agent_async(
        "I need some fitness motivation hashtags.",
        runner=runner,
        user_id=USER_ID,
        session_id=SESSION_ID
    )

    await call_agent_async(
        "Suggest hashtags for my youtube video on AI agents.",
        runner=runner,
        user_id=USER_ID,
        session_id=SESSION_ID
    )
    await call_agent_async(
        "Suggest  for some bad stuff.",
        runner=runner,
        user_id=USER_ID,
        session_id=SESSION_ID
    )


In [ ]:
await run_conversation()


>>> User Query: Generate hashtags for travel photography.


--- Smart Hashtag Generator called for: travel photography ---
<<< Agent Response: Okay! Here are some hashtags for travel photography: #travel, #photography.

>>> User Query: I need some fitness motivation hashtags.


--- Smart Hashtag Generator called for: fitness motivation ---
<<< Agent Response: Here are some fitness motivation hashtags: #fitness, #motivation.


>>> User Query: Suggest hashtags for my youtube video on AI agents.


--- Smart Hashtag Generator called for: youtube video on AI agents ---
<<< Agent Response: Here are some hashtags for your youtube video on AI agents: #youtube, #video, #on, #ai, #agents.


>>> User Query: Suggest  for some bad stuff.
<<< Agent Response: I am sorry, I cannot fulfill that request. I am not supposed to generate hashtags for bad stuff.

